In [1]:
# Install only the essentials for RAG (stable in Colab)
# transformers==4.x is used because it supports text2text-generation (Flan-T5)
!pip -q install faiss-cpu sentence-transformers "transformers==4.44.2" accelerate streamlit

print("Install complete ✅")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.1 MB/s eta 0:00:00
Install complete ✅


In [2]:
# Core tools
import numpy as np
import faiss

# Embeddings model
from sentence_transformers import SentenceTransformer

# LLM pipeline (Flan-T5 works well for instruction-style answers)
from transformers import pipeline

# Optional: check GPU
import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: True


In [3]:
# Custom corpus (Knowledge Base) for RAG Chatbot
# Requirement: "Custom corpus (Wikipedia pages, internal docs, or any knowledge base)"
# We are creating a mini knowledge base related to AI/ML internship topics.

documents = [
    {
        "title": "Internship Overview and Goals",
        "text": """
DevelopersHub AI/ML internship focuses on practical machine learning skills such as:
data preprocessing, model training, evaluation, and basic deployment.
For advanced tasks, focus areas include transformers, ML pipelines, RAG chatbots, and LLM applications.
The goal is to build real-world, portfolio-ready projects.
"""
    },
    {
        "title": "What is Retrieval-Augmented Generation (RAG)?",
        "text": """
Retrieval-Augmented Generation (RAG) is a technique that improves chatbot accuracy.
It works in two stages:
1) Retrieval: search a document knowledge base to find the most relevant text chunks.
2) Generation: use a language model to generate an answer using the retrieved text as context.
RAG helps reduce hallucination because answers are grounded in retrieved documents.
"""
    },
    {
        "title": "Vector Embeddings and Similarity Search",
        "text": """
Embeddings convert text into numerical vectors that represent meaning.
Texts with similar meaning have embeddings that are close in vector space.
Similarity search finds the closest vectors to a query vector.
This is useful for retrieving relevant document chunks for RAG systems.
"""
    },
    {
        "title": "FAISS Vector Store (Document Indexing)",
        "text": """
FAISS (Facebook AI Similarity Search) is a library for fast similarity search on vectors.
In RAG, we store embeddings of document chunks in FAISS.
When a user asks a question, we embed the query and retrieve the most similar chunks from FAISS.
This provides fast and efficient document retrieval.
"""
    },
    {
        "title": "Conversation Memory in Chatbots",
        "text": """
Conversation memory allows a chatbot to remember previous messages.
This is important for follow-up questions such as:
User: What is RAG?
User: How does it reduce hallucinations?
The second question depends on the first, so memory helps maintain context.
A simple memory approach is storing the last few user-bot messages and including them in the prompt.
"""
    }
]

print("Docs loaded:", len(documents))
print("Sample title:", documents[0]["title"])


Docs loaded: 5
Sample title: Internship Overview and Goals


In [4]:
# Convert document dicts into chunkable text
raw_texts = [f"Title: {d['title']}\n{d['text']}" for d in documents]

# Simple chunking function (keeps it beginner-friendly)
def chunk_text(text, chunk_size=350, overlap=60):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

# Create chunks
chunks = []
for t in raw_texts:
    chunks.extend(chunk_text(t))

print("Total chunks:", len(chunks))
print("Sample chunk:\n", chunks[0][:250])


Total chunks: 10
Sample chunk:
 Title: Internship Overview and Goals

DevelopersHub AI/ML internship focuses on practical machine learning skills such as:
data preprocessing, model training, evaluation, and basic deployment.
For advanced tasks, focus areas include transformers, ML 


In [5]:
# Load embedding model (fast + good for RAG)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Create embeddings for all chunks
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True).astype("float32")

# Build FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index built ✅")
print("Embedding dimension:", dimension)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built ✅
Embedding dimension: 384


In [6]:
# Flan-T5 is a text2text model (good for instruction answers)
generator = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM loaded ✅")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


LLM loaded ✅


In [7]:
def retrieve_chunks(query, top_k=3):
    """
    Retrieve top_k most relevant chunks from FAISS using embedding similarity.
    """
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    retrieved = [chunks[i] for i in indices[0]]
    return retrieved


In [8]:
# Conversation memory (stores last turns)
chat_history = []

def rag_answer(user_question):
    """
    Full RAG pipeline:
    1) Retrieve relevant chunks
    2) Build prompt with retrieved context + chat history
    3) Generate final answer using LLM
    """
    retrieved = retrieve_chunks(user_question, top_k=3)

    # Keep memory short so prompt does not become too long
    memory_text = "\n".join(chat_history[-4:])

    # Create context from retrieved chunks
    context_text = "\n\n".join(retrieved)

    # Prompt engineering for safety + grounding
    prompt = f"""
You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say: "I don't know based on the documents."

Conversation history:
{memory_text}

Context:
{context_text}

Question: {user_question}
Answer:
""".strip()

    # Generate answer
    output = generator(prompt)[0]["generated_text"]

    # Update memory
    chat_history.append(f"User: {user_question}")
    chat_history.append(f"Bot: {output}")

    return output, retrieved


In [9]:
a1, r1 = rag_answer("What is RAG?")
print("Answer 1:", a1)

a2, r2 = rag_answer("How does it reduce hallucinations?")
print("\nAnswer 2:", a2)

print("\nRetrieved chunks for Q2:")
for c in r2:
    print("-", c[:120], "...")


Answer 1: a technique that improves chatbot accuracy

Answer 2: RAG helps reduce hallucination because answers are grounded in retrieved documents.

Retrieved chunks for Q2:
- Title: Conversation Memory in Chatbots

Conversation memory allows a chatbot to remember previous messages.
This is impo ...
- erate an answer using the retrieved text as context.
RAG helps reduce hallucination because answers are grounded in retr ...
- evant document chunks for RAG systems.
 ...


In [10]:
app_code = r'''
import streamlit as st
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

st.set_page_config(page_title="RAG Chatbot", layout="centered")
st.title("Context-Aware RAG Chatbot (FAISS + Embeddings + LLM)")
st.write("This chatbot retrieves answers from a custom document store and remembers conversation context.")

documents = [
    {"title": "RAG Basics", "text": "RAG retrieves relevant chunks first, then generates answers using retrieved context to reduce hallucinations."},
    {"title": "FAISS Vector Search", "text": "FAISS stores embeddings and retrieves the most similar chunks for a question using vector search."},
    {"title": "Conversation Memory", "text": "Memory stores chat history so follow-up questions can be answered using previous context."}
]

raw_texts = [f"Title: {d['title']}\n{d['text']}" for d in documents]

def chunk_text(text, chunk_size=350, overlap=60):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = []
for t in raw_texts:
    chunks.extend(chunk_text(t))

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

generator = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=200)

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

def retrieve_chunks(query, top_k=3):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    return [chunks[i] for i in indices[0]]

def rag_answer(user_question):
    retrieved = retrieve_chunks(user_question, top_k=3)
    memory_text = "\n".join(st.session_state.chat_history[-4:])
    context_text = "\n\n".join(retrieved)

    prompt = f"""
You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say: "I don't know based on the documents."

Conversation history:
{memory_text}

Context:
{context_text}

Question: {user_question}
Answer:
""".strip()

    output = generator(prompt)[0]["generated_text"]

    st.session_state.chat_history.append(f"User: {user_question}")
    st.session_state.chat_history.append(f"Bot: {output}")

    return output

q = st.text_input("Ask a question:")

if st.button("Send") and q.strip():
    answer = rag_answer(q)
    st.success(answer)

st.write("### Conversation")
for line in st.session_state.chat_history:
    st.write(line)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created ✅")


app.py created ✅


## Objective

The objective of this task is to build a context-aware conversational chatbot using the
Retrieval-Augmented Generation (RAG) approach. The chatbot is designed to retrieve relevant
information from a custom document knowledge base and generate accurate responses based on
the retrieved content.

This task involves converting textual documents into vector embeddings, storing them in a
vector database (FAISS), and retrieving the most relevant document chunks for user queries.
Additionally, a simple conversation memory mechanism is implemented to maintain the context
of previous interactions, allowing the chatbot to respond effectively to follow-up questions.

The final system demonstrates the integration of document retrieval, contextual memory, and
language model-based answer generation for building intelligent conversational AI systems.


## Conclusion

In this task, a context-aware RAG-based chatbot was successfully developed using a custom
document corpus. The documents were split into smaller chunks and transformed into vector
embeddings using a sentence-transformer model. These embeddings were stored in a FAISS
vector index to enable efficient similarity-based retrieval of relevant information.

During user interaction, the chatbot retrieves the most relevant document chunks based on
the query and generates responses using a pre-trained language model. A basic conversation
memory mechanism was also implemented to retain chat history and improve the handling of
follow-up questions.

This approach helps reduce hallucinations by grounding responses in retrieved documents
and demonstrates the practical implementation of Retrieval-Augmented Generation (RAG) for
building context-aware conversational AI applications.
